# Module 05 — R-CNN Family (SOLUTIONS)

In [ ]:
import torch
import math

def encode_boxes(anchors, gt_boxes):
    """Encode GT boxes as deltas relative to anchors."""
    # Convert xyxy → cxcywh
    aw = anchors[:,2] - anchors[:,0]; ah = anchors[:,3] - anchors[:,1]
    ax = anchors[:,0] + aw/2;        ay = anchors[:,1] + ah/2
    gw = gt_boxes[:,2] - gt_boxes[:,0]; gh = gt_boxes[:,3] - gt_boxes[:,1]
    gx = gt_boxes[:,0] + gw/2;         gy = gt_boxes[:,1] + gh/2
    dx = (gx - ax) / aw
    dy = (gy - ay) / ah
    dw = torch.log(gw / aw)
    dh = torch.log(gh / ah)
    return torch.stack([dx, dy, dw, dh], dim=1)

def decode_boxes(anchors, deltas):
    """Decode box deltas to xyxy boxes."""
    aw = anchors[:,2] - anchors[:,0]; ah = anchors[:,3] - anchors[:,1]
    ax = anchors[:,0] + aw/2;        ay = anchors[:,1] + ah/2
    px = ax + deltas[:,0] * aw
    py = ay + deltas[:,1] * ah
    pw = aw * torch.exp(deltas[:,2])
    ph = ah * torch.exp(deltas[:,3])
    return torch.stack([px - pw/2, py - ph/2, px + pw/2, py + ph/2], dim=1)

# Round-trip test
anchors = torch.tensor([[10.,10.,50.,50.],[100.,80.,200.,160.]])
gt      = torch.tensor([[15.,12.,55.,48.],[110.,90.,195.,155.]])
deltas  = encode_boxes(anchors, gt)
decoded = decode_boxes(anchors, deltas)
print('Max decode error:', (decoded - gt).abs().max().item())